# WT103 Learning Curves And Scaling

This notebook is for the current WT103 scaling runs.

Practical plotting choices:

- Learning curves: one panel per model family, `val_ppl` vs epoch, log y-axis, color by recurrent core scale.
- Scaling law: x-axis should be recurrent core params, not total params.
- Report best-checkpoint test PPL, not final-checkpoint test PPL, for the scaling fit.
- With only a few completed scales, use a simple log-log power fit first. Offset fits such as `L_inf + A N^{-alpha}` are better once you have more scales or more seeds.

This notebook is written to be safe in a headless backend and to degrade cleanly when only part of the scaling sweep is finished.

In [ ]:
from __future__ import annotations

import json
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_ROOT = ROOT / 'runs' / 'wikitext-103-raw-v1'
SCALE_SUFFIX = '_scale_rmsnorm'

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)


def render_fig(fig):
    display(fig)
    plt.close(fig)


def note(text: str):
    display(Markdown(text))

In [ ]:
def _read_json(path: Path):
    return json.loads(path.read_text())


def _read_history(path: Path) -> pd.DataFrame:
    if not path.exists() or not path.read_text().strip():
        return pd.DataFrame(columns=['epoch', 'train_ce', 'val_ppl', 'best_val_ppl', 'global_step'])
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return pd.DataFrame(rows)


@lru_cache(maxsize=None)
def _load_cfg(cfg_path: str) -> dict:
    import yaml
    return yaml.safe_load(Path(cfg_path).read_text()) or {}


@lru_cache(maxsize=None)
def _param_breakdown_from_cfg(cfg_path: str) -> dict:
    from scripts._common import ensure_repo_root_on_path, build_model, count_params
    ensure_repo_root_on_path()
    cfg = _load_cfg(cfg_path)
    model_name = str(cfg['model_name']).strip().lower()
    model = build_model(cfg, model_name, backend_name='torch')
    total = count_params(model, backend_name='torch')
    if model_name == 'neo':
        emb = model.emb.weight.numel()
        core = sum(p.numel() for p in model.recurrent.parameters() if p.requires_grad)
        proj = sum(p.numel() for p in model.in_proj.parameters()) if hasattr(model.in_proj, 'parameters') else 0
        proj += sum(p.numel() for p in model.out_proj.parameters()) if hasattr(model.out_proj, 'parameters') else 0
        head = model.output_bias.numel() if model.output_bias is not None else 0
        if model.head is not None:
            head += sum(p.numel() for p in model.head.parameters() if p.requires_grad)
        return {'total': total, 'embeddings': emb, 'core': core, 'head_proj': proj + head}
    if model_name == 'lstm':
        emb = model.emb.weight.numel()
        core = sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
        proj = sum(p.numel() for p in model.in_proj.parameters()) if hasattr(model.in_proj, 'parameters') else 0
        proj += sum(p.numel() for p in model.out_proj.parameters()) if hasattr(model.out_proj, 'parameters') else 0
        head = model.output_bias.numel() if model.output_bias is not None else 0
        if model.head is not None:
            head += sum(p.numel() for p in model.head.parameters() if p.requires_grad)
        return {'total': total, 'embeddings': emb, 'core': core, 'head_proj': proj + head}
    raise ValueError(f'Unsupported model_name for this notebook: {model_name}')


def _core_label(core_params: float) -> str:
    core_m = core_params / 1e6
    if core_m < 15:
        return '~10M'
    if core_m < 30:
        return '~20M'
    return '~40M'


def collect_latest_scale_runs() -> pd.DataFrame:
    rows = []
    for metrics_path in RUNS_ROOT.glob('*/*/*/metrics.json'):
        run_dir = metrics_path.parent
        run_tag = run_dir.parent.name
        if not run_tag.endswith(SCALE_SUFFIX):
            continue
        model_name = run_dir.parent.parent.name
        if model_name not in {'neo', 'lstm'}:
            continue
        cfg_path = run_dir / 'config.yaml'
        hist_path = run_dir / 'history.jsonl'
        if not cfg_path.exists():
            continue
        metrics = _read_json(metrics_path)
        cfg = _load_cfg(str(cfg_path))
        hist = _read_history(hist_path)
        breakdown = _param_breakdown_from_cfg(str(cfg_path))
        val_series = hist['val_ppl'].astype(float) if ('val_ppl' in hist.columns and not hist.empty) else pd.Series(dtype=float)
        best_epoch = int(hist.loc[val_series.idxmin(), 'epoch']) if not val_series.empty else None
        final_epoch = int(hist['epoch'].max()) if ('epoch' in hist.columns and not hist.empty) else None
        rows.append({
            'run_dir': str(run_dir),
            'timestamp': run_dir.name,
            'run_tag': run_tag,
            'model_name': model_name,
            'config_path': str(cfg_path),
            'history_path': str(hist_path),
            'n_layers': int(cfg.get('n_layers', 0)),
            'd_model': int(cfg.get('d_model', 0)),
            'epochs_seen': int(len(hist)),
            'best_epoch': best_epoch,
            'final_epoch': final_epoch,
            'best_val_ppl': float(val_series.min()) if not val_series.empty else metrics.get('val_ppl'),
            'final_val_ppl': float(val_series.iloc[-1]) if not val_series.empty else metrics.get('val_ppl'),
            'final_test_ppl': metrics.get('test_ppl'),
            'best_test_ppl': metrics.get('best_ckpt_test_ppl_torch', metrics.get('test_ppl')),
            'total_params': breakdown['total'],
            'embedding_params': breakdown['embeddings'],
            'core_params': breakdown['core'],
            'head_proj_params': breakdown['head_proj'],
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df.sort_values(['run_tag', 'timestamp']).groupby('run_tag', as_index=False).tail(1).copy()
    df['core_label'] = df['core_params'].map(_core_label)
    df['core_params_m'] = df['core_params'] / 1e6
    df['total_params_m'] = df['total_params'] / 1e6
    df['embedding_params_m'] = df['embedding_params'] / 1e6
    return df.sort_values(['model_name', 'core_params']).reset_index(drop=True)


scale_df = collect_latest_scale_runs()
if scale_df.empty:
    note('**No completed scaling runs found yet.** Run the scaling experiments first, then re-run this notebook.')
scale_df

## What To Visualize

For the current project, the cleanest story is:

- learning dynamics: validation PPL vs epoch
- scaling behavior: best-checkpoint test PPL vs recurrent core params
- model-family comparison: Neo and LSTM on the same core-scale axis

The recurrent core params are the right x-axis because total params are partly dominated by embeddings, while your intended comparison is mainly about recurrent capacity.

In [ ]:
COLOR_MAP = {'~10M': '#4C78A8', '~20M': '#F58518', '~40M': '#54A24B'}

if not scale_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for ax, model_name in zip(axes, ['neo', 'lstm']):
        sub = scale_df[scale_df['model_name'] == model_name].sort_values('core_params')
        for _, row in sub.iterrows():
            hist = _read_history(Path(row['history_path']))
            if hist.empty or 'val_ppl' not in hist.columns:
                continue
            color = COLOR_MAP.get(row['core_label'], '#333333')
            label = f"{row['core_label']} core ({row['core_params_m']:.1f}M)"
            ax.plot(hist['epoch'], hist['val_ppl'], marker='o', lw=2, color=color, label=label)
            best_idx = hist['val_ppl'].astype(float).idxmin()
            ax.scatter(
                hist.loc[best_idx, 'epoch'],
                hist.loc[best_idx, 'val_ppl'],
                color=color,
                s=90,
                marker='*',
                edgecolor='black',
                linewidth=0.6,
            )
        ax.set_title(model_name.upper())
        ax.set_xlabel('Epoch')
        ax.set_yscale('log')
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(frameon=True)
        else:
            ax.text(0.5, 0.5, 'No finished curves yet', ha='center', va='center', transform=ax.transAxes)
    axes[0].set_ylabel('Validation PPL (log scale)')
    fig.suptitle('WT103 Learning Curves For Scaling Runs', y=1.03, fontsize=14)
    plt.tight_layout()
    render_fig(fig)

In [ ]:
summary_cols = [
    'model_name', 'run_tag', 'n_layers', 'd_model', 'epochs_seen', 'core_params_m',
    'best_epoch', 'best_val_ppl', 'best_test_ppl', 'final_test_ppl', 'total_params_m'
]
if not scale_df.empty:
    display(scale_df[summary_cols].sort_values(['model_name', 'core_params_m']).reset_index(drop=True))

In [ ]:
def fit_loglog_power(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    x = x[mask]
    y = y[mask]
    if x.size < 2:
        return None
    coef = np.polyfit(np.log(x), np.log(y), 1)
    slope, intercept = coef
    return {
        'slope': float(slope),
        'intercept': float(intercept),
        'alpha_if_decreasing': float(-slope),
        'predict': lambda x_new: np.exp(intercept) * np.power(np.asarray(x_new, dtype=float), slope),
    }


fit_rows = []
if not scale_df.empty:
    for model_name, sub in scale_df.groupby('model_name'):
        fit = fit_loglog_power(sub['core_params_m'].values, sub['best_test_ppl'].values)
        fit_rows.append({
            'model_name': model_name,
            'n_points': int(len(sub)),
            'fit_available': fit is not None,
            'slope_loglog': np.nan if fit is None else fit['slope'],
            'alpha_if_decreasing': np.nan if fit is None else fit['alpha_if_decreasing'],
        })
fit_df = pd.DataFrame(fit_rows)
fit_df

In [ ]:
if not scale_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x_grid = np.geomspace(scale_df['core_params_m'].min() * 0.9, scale_df['core_params_m'].max() * 1.1, 200)

    for model_name, sub in scale_df.groupby('model_name'):
        sub = sub.sort_values('core_params_m')
        color = '#4C78A8' if model_name == 'neo' else '#F58518'
        fit = fit_loglog_power(sub['core_params_m'].values, sub['best_test_ppl'].values)

        axes[0].plot(sub['core_params_m'], sub['best_test_ppl'], marker='o', lw=0, ms=8, color=color, label=model_name.upper())
        axes[1].plot(sub['core_params_m'], sub['best_test_ppl'], marker='o', lw=0, ms=8, color=color, label=model_name.upper())

        if fit is not None:
            y_fit = fit['predict'](x_grid)
            axes[0].plot(x_grid, y_fit, lw=2, color=color, alpha=0.9)
            axes[1].plot(x_grid, y_fit, lw=2, color=color, alpha=0.9)

    for _, row in scale_df.iterrows():
        axes[0].annotate(row['core_label'], (row['core_params_m'], row['best_test_ppl']), textcoords='offset points', xytext=(5, 6), fontsize=9)
        axes[1].annotate(row['core_label'], (row['core_params_m'], row['best_test_ppl']), textcoords='offset points', xytext=(5, 6), fontsize=9)

    axes[0].set_xlabel('Recurrent Core Params (Millions)')
    axes[0].set_ylabel('Best-Checkpoint Test PPL')
    axes[0].set_title('Readable Scaling Plot')
    axes[0].legend(frameon=True)

    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Recurrent Core Params (Millions, log scale)')
    axes[1].set_ylabel('Best-Checkpoint Test PPL (log scale)')
    axes[1].set_title('Log-Log Power Fit')
    axes[1].legend(frameon=True)

    fig.suptitle('WT103 Scaling Law On Best-Checkpoint Test PPL', y=1.03, fontsize=14)
    plt.tight_layout()
    render_fig(fig)
else:
    note('**Scaling plot skipped** because no finished scaling runs were found.')

## Fitting Advice

Right now the correct default fit is the notebook's log-log power fit.

- Use it to estimate the slope and compare Neo vs LSTM.
- Treat the slope as provisional if there are only 2 to 3 completed scales.
- Once more scales or seeds are available, upgrade to an offset fit such as `L(N) = L_inf + A N^{-alpha}` and compare residuals.

For the current stage of the project, the best presentation is usually:

- Figure 1: learning curves
- Figure 2: readable scaling scatter
- Figure 3: log-log fit panel
- Table 1: best epoch, best validation PPL, best-checkpoint test PPL, core params